# Export native final GRN edges

Run from the package root (`Path.cwd()`). This notebook exports the six historical native GRN results in `data/grn/`; it does not train a GRN or access H5AD files.

Weights follow `pyscenic.utils.load_motifs → pyscenic.prune.df2regulons → gene2weight`. Native `regulons.json` determines exact final memberships; the motifs determine TF identity and signed weights. Self edges and negative weights are retained. No raw adjacency input or invented edge statistics are used. The weights are native motif-pruning values, not newly estimated effects.

Existing cell outputs retain the previous execution record. The export summary and validation are displayed in the notebook.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display
from pyscenic.utils import load_motifs
from pyscenic.prune import df2regulons

PACKAGE = Path.cwd()
RUNS = ("full", "rep01", "rep02", "rep03", "rep04", "rep05")
export_started_utc = datetime.now(timezone.utc).isoformat()
sources = pd.DataFrame([
    {"run": run, "role": role, "package_path": str(path.relative_to(PACKAGE)),
     "package_bytes": path.stat().st_size}
    for run in RUNS
    for role, filename in [("regulons", "regulons.json"), ("motifs", "motifs.csv")]
    for path in [PACKAGE / "data/grn" / run / filename]
])
display(sources)


,run,role,package_path,package_bytes
0,full,regulons,data/grn/full/regulons.json,1193006
1,full,motifs,data/grn/full/motifs.csv,8040190
2,rep01,regulons,data/grn/rep01/regulons.json,580286
3,rep01,motifs,data/grn/rep01/motifs.csv,5909175
4,rep02,regulons,data/grn/rep02/regulons.json,579313
5,rep02,motifs,data/grn/rep02/motifs.csv,5690254
6,rep03,regulons,data/grn/rep03/regulons.json,588212
7,rep03,motifs,data/grn/rep03/motifs.csv,5792808
8,rep04,regulons,data/grn/rep04/regulons.json,582270
9,rep04,motifs,data/grn/rep04/motifs.csv,5979840


In [2]:
export_rows = []
for run in RUNS:
    directory = PACKAGE / "data/grn" / run
    native = json.loads((directory / "regulons.json").read_text())
    weighted = {
        regulon.name: (regulon.transcription_factor, dict(regulon.gene2weight))
        for regulon in df2regulons(load_motifs(str(directory / "motifs.csv")))
    }
    records = []
    for name, targets in native.items():
        assert len(targets) == len(set(targets)), f"Duplicate native targets: {run} {name}"
        tf, weights = weighted[name]
        for target in targets:
            records.append((run, name, tf, target, weights[target]))
    edges = pd.DataFrame(records, columns=["run", "regulon", "TF", "target", "weight"])
    assert not edges.duplicated(["run", "regulon", "target"]).any()
    assert np.isfinite(edges["weight"]).all()
    assert set(zip(edges["regulon"], edges["target"])) == {
        (name, target) for name, targets in native.items() for target in targets
    }
    edges = edges.sort_values(["regulon", "target"]).reset_index(drop=True)
    destination = directory / "final_edges.tsv.gz"
    edges.to_csv(destination, sep="\t", index=False, compression={"method": "gzip", "mtime": 0})
    reread = pd.read_csv(destination, sep="\t", float_precision="round_trip")
    pd.testing.assert_frame_equal(edges, reread, check_exact=True)
    assert destination.stat().st_size < 10 * 1024**2
    export_rows.append({
        "run": run, "n_regulons": len(native), "n_edges": len(edges),
        "self_edges": int((edges["TF"] == edges["target"]).sum()),
        "negative_weights": int((edges["weight"] < 0).sum()),
        "zero_weights": int((edges["weight"] == 0).sum()),
        "min_weight": float(edges["weight"].min()),
        "max_weight": float(edges["weight"].max()),
        "output": str(destination.relative_to(PACKAGE)),
        "bytes": destination.stat().st_size,
        "sha256": hashlib.sha256(destination.read_bytes()).hexdigest(),
        "native_membership_exact": True, "weights_roundtrip_exact": True,
    })
export_summary = pd.DataFrame(export_rows)
assert tuple(export_summary["n_regulons"]) == (260, 220, 221, 213, 218, 212)
assert tuple(export_summary["n_edges"]) == (68492, 33270, 33212, 33745, 33381, 34063)
display(export_summary.drop(columns=["sha256"]))

Create regulons from a dataframe of enriched features.
Additional columns saved: []


Create regulons from a dataframe of enriched features.
Additional columns saved: []


Create regulons from a dataframe of enriched features.
Additional columns saved: []


Create regulons from a dataframe of enriched features.
Additional columns saved: []


Create regulons from a dataframe of enriched features.
Additional columns saved: []


Create regulons from a dataframe of enriched features.
Additional columns saved: []


,run,n_regulons,n_edges,self_edges,negative_weights,zero_weights,min_weight,max_weight,output,bytes,native_membership_exact,weights_roundtrip_exact
0,full,260,68492,112,13,0,-0.002940,1.0,data/grn/full/final_edges.tsv.gz,960885,True,True
1,rep01,220,33270,90,15,0,-0.008146,1.0,data/grn/rep01/final_edges.tsv.gz,468929,True,True
2,rep02,221,33212,93,8,0,-0.002399,1.0,data/grn/rep02/final_edges.tsv.gz,468405,True,True
3,rep03,213,33745,90,14,0,-0.002111,1.0,data/grn/rep03/final_edges.tsv.gz,474986,True,True
4,rep04,218,33381,90,14,0,-0.001537,1.0,data/grn/rep04/final_edges.tsv.gz,470307,True,True
5,rep05,212,34063,82,14,0,-0.003791,1.0,data/grn/rep05/final_edges.tsv.gz,480600,True,True


In [3]:
export_finished_utc = datetime.now(timezone.utc).isoformat()
validation = {
    "schema_version": 1,
    "export": {
        "status": "passed", "actual_execution": True,
        "started_utc": export_started_utc, "finished_utc": export_finished_utc,
        "python": sys.executable,
        "pyscenic_version": importlib.metadata.version("pyscenic"),
        "source_results": "historical native JSON and motif-pruning CSV in data/grn",
        "method": "load_motifs -> df2regulons -> gene2weight, restricted exactly to native JSON memberships",
        "self_edges": "retained", "negative_weights": "retained",
        "raw_adjacency_used": False, "training_performed": False,
        "rows": export_rows,
    },
    "offline_analysis": {"status": "pending", "notebook": "notebooks/analysis/02_network_topk.ipynb"},
}
display(pd.DataFrame([{
    "status": validation["export"]["status"],
    "actual_execution": True, "started_utc": export_started_utc,
    "finished_utc": export_finished_utc,
    "python": sys.executable, "pyscenic_version": validation["export"]["pyscenic_version"],
    "training_performed": False,
}]))

,status,actual_execution,started_utc,finished_utc,python,pyscenic_version,training_performed
0,passed,True,2026-11-07T16:16:41.062678+00:00,2026-11-07T16:16:51.219632+00:00,/data/hulei/HongtaoYu_Lab/Wenyanran/spaGRN/bin...,0.12.1,False
